In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)

RgObs={}


In [ ]:

DstVgrid='L93'

DstInfo = GrU.gridInfo('fv1x1') #,Vgrid=DstVgrid)
lat_a,lon_a = GrU.latlon( scrip= DstInfo['scrip'], Hkey=DstInfo['Hkey'] )

DstVgridFile = GrU.gridInfo(Vgrid=DstVgrid,VgridOnly=True) #,Vgrid=DstVgrid)
vDst=xr.open_dataset( DstVgridFile )
amid_Dst = vDst['hyam'].values
bmid_Dst = vDst['hybm'].values
aint_Dst = vDst['hyai'].values
bint_Dst = vDst['hybi'].values
lev_Dst  = 1_000.*( amid_Dst + bmid_Dst )
ilev_Dst  = 1_000.*( aint_Dst + bint_Dst )

hyai = aint_Dst
hybi = bint_Dst
hyam = amid_Dst
hybm = bmid_Dst


SrcVgridFile = GrU.gridInfo(Vgrid='ERA5',VgridOnly=True) 
vSrc=xr.open_dataset( SrcVgridFile )
amid_Src = vSrc['hyam'].values
bmid_Src = vSrc['hybm'].values
aint_Src = vSrc['hyai'].values
bint_Src = vSrc['hybi'].values
lev_Src  = 1_000.*( amid_Src + bmid_Src )

# def Vert(DstVgrid=None, DstTZHkey=None, SrcVgrid=None, xfld_Src=None, ps_Src=None, pmid_output=False, long_output=False , verbose=False ):


Dir_in  = f'/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_monthly/ZYX'
Dir_out = f'/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_{DstVgrid}_monthly/ZYX'
#######
os.makedirs( Dir_out , exist_ok=True )
        
#for y in np.arange( start=2015, stop=2025 ):
#    for m in np.arange( start=1, stop=13 ):
for y in np.arange( start=2020, stop=2025 ):
    for m in np.arange( start=1, stop=13 ):
        
        
        file_in  = f'ERA5.fv1x1_zERA.time.{str(y).zfill(4)}-{str(m).zfill(2)}.nc'
        file_out = f'ERA5.fv1x1_{DstVgrid}.time.{str(y).zfill(4)}-{str(m).zfill(2)}.nc'
        X = xr.open_dataset( f'{Dir_in}/{file_in}' )
        
        print( f'Processing {file_in}')

        ps=X.PS.values
        
        u=X.U.values
        u_z =RgF.Vert(xfld_Src=u , SrcVgrid='ERA5', DstVgrid=DstVgrid, DstTZHkey='tzyx', ps_Src=ps )
        print( '   .. U' )

        v=X.V.values
        v_z =RgF.Vert(xfld_Src=v , SrcVgrid='ERA5', DstVgrid=DstVgrid, DstTZHkey='tzyx', ps_Src=ps )
        print( '   .. V' )

        q=X.Q.values
        q_z =RgF.Vert(xfld_Src=q , SrcVgrid='ERA5', DstVgrid=DstVgrid, DstTZHkey='tzyx', ps_Src=ps )
        print( '   .. Q' )

        te=X.T.values
        te_z =RgF.Vert(xfld_Src=te , SrcVgrid='ERA5', DstVgrid=DstVgrid, DstTZHkey='tzyx', ps_Src=ps )
        print( '   .. T' )

        """
        ## Oooops .... forgot this ...
        og=X.OMEGA.values
        og_z =RgF.Vert(xfld_Src=og , SrcVgrid='ERA5', DstVgrid=DstVgrid, DstTZHkey='tzyx', ps_Src=ps )
        print( '   .. OMEGA' )
        """


        #ilev_Dst = ( aint_Dst + bint_Dst )* 1_000. #* 100_000.
        #lev_Dst  = ( amid_Dst + bmmif_Dst )* 1_000. #* 100_000.
        
        coords = dict( 
            lon  = ( ["lon"],lon_a ),
            lat  = ( ["lat"],lat_a ),
            lev  = ( ["lev"],lev_Dst),
            ilev = ( ["ilev"],ilev_Dst),
            time = ( ["time"],  X.time.values ) )
        
        Wds = xr.Dataset( coords=coords  )
        
        Dar = xr.DataArray( data=hyai, dims=('ilev',),
                            attrs=dict( description='interface hybrid eta coordinate A-coeff ',units='1',) ,) 
        Wds['hyai'] = Dar
        
        Dar = xr.DataArray( data=hybi, dims=('ilev',),
                            attrs=dict( description='interface hybrid eta coordinate B-coeff ',units='1',) ,) 
        Wds['hybi'] = Dar
        
        Dar = xr.DataArray( data=hyam, dims=('lev',),
                            attrs=dict( description='mid-level hybrid eta coordinate A-coeff ',units='1',) ,) 
        Wds['hyam'] = Dar
        
        Dar = xr.DataArray( data=hybm, dims=('lev',),
                            attrs=dict( description='mid-level hybrid eta coordinate B-coeff ',units='1',) ,) 
        Wds['hybm'] = Dar
        


        Dar = xr.DataArray( data=te_z , 
                            dims=('time','lev','lat','lon',),
                            attrs=dict( description='Air Temperature',units='K',) ,) 
        Wds['T'] = Dar
        
        Dar = xr.DataArray( data=q_z , 
                            dims=('time','lev','lat','lon',),
                            attrs=dict( description='specific humidity',units='kg kg-1',) ,) 
        Wds['Q'] = Dar
        
        Dar = xr.DataArray( data=u_z , 
                            dims=('time','lev','lat','lon',),
                            attrs=dict( description='X-wind',units='m s-1',) ,) 
        Wds['U'] = Dar
        
        Dar = xr.DataArray( data=v_z , 
                            dims=('time','lev','lat','lon',),
                            attrs=dict( description='Y-wind',units='m s-1',) ,) 
        Wds['V'] = Dar
        
        Dar = xr.DataArray( data=ps , 
                            dims=('time','lat','lon',),
                            attrs=dict( description='Surface Pressure',units='Pa',) ,) 
        Wds['PS'] = Dar
        

        Wds.to_netcdf( f'{Dir_out}/{file_out}')
        print(f'      Wrote {Dir_out}/{file_out} ' )